# semaine 6

extract data

In [ ]:
import pandas as pd

df = pd.read_csv('microdados_ed_basica_2024_SP.csv', sep=';', encoding='latin1', low_memory=False)

print(df.info())
print(df.head())

In [ ]:

import pandas as pd
import numpy as np
from sklearn.linear_model import LinearRegression
import networkx as nx
from sklearn.impute import SimpleImputer
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler



def graph(lien, cols):
    df = pd.read_csv(lien, sep=';', encoding='latin1', low_memory=False)

    imputer = SimpleImputer(strategy='median')
    X_clean = pd.DataFrame(imputer.fit_transform(df[cols]), columns=cols)
    scaler = StandardScaler()
    X_scaled = pd.DataFrame(scaler.fit_transform(X_clean), columns=cols)


    binary_data = X_clean[[c for c in cols if c.startswith('IN_')]]
    non_binary_data = X_clean[[c for c in cols if not c.startswith('IN_')]]


    #rex
    def rex_algorithm(data_df):
        X = data_df.values.copy()
        names = data_df.columns.tolist()
        n = len(names)
        remaining = list(range(n))
        order = []
        
        #iterate
        while len(remaining) > 0:
            scores = []
            for i in remaining:
                total_dep= 0
                xi = X[:, i].reshape(-1, 1)
                for j in remaining:
                    if i == j: continue
                    xj = X[:, j].reshape(-1, 1)
                    lr = LinearRegression().fit(xi,xj)
                    residual = xj - lr.predict(xi)
                    total_dep += abs(np.corrcoef(xi.flatten(), residual.flatten())[0,1])
                scores.append((total_dep, i))
            
            scores.sort()
            best_root = scores[0][1]
            order.append(best_root)
            remaining.remove(best_root)
            if len(remaining) >0:
                xi_root = X[:, best_root].reshape(-1, 1)
                for j in remaining:
                    xj = X[:,j].reshape(-1, 1)
                    lr = LinearRegression().fit(xi_root, xj)
                    X[:, j] = (xj - lr.predict(xi_root)).flatten()
                    
        return [names[i] for i in order]

    causal_order = rex_algorithm(X_scaled)

    #Construct DAG X = BX + e
    B = pd.DataFrame(0.0, index=cols, columns=cols)
    X_std_df = pd.DataFrame(X_scaled, columns=cols)

    for i, target in enumerate(causal_order):
        if i == 0: continue
        parents = causal_order[:i]
        
        y = X_std_df[target]
        X_p = X_std_df[parents]
        
        reg = LinearRegression().fit(X_scaled[parents], X_scaled[target])
        for p_idx, p_name in enumerate(parents):
            B.loc[target, p_name] = reg.coef_[p_idx]


    B_pruned= B.copy()
    B_pruned[np.abs(B_pruned) < 0.05] =0

    dico = {node: [] for node in cols}
    G= nx.DiGraph()
    for node in cols:
        G.add_node(node)
        

    for row in B_pruned.index:
        for col in B_pruned.columns:
            if B_pruned.loc[row, col]!= 0:
                G.add_edge(col, row, weight=round(B_pruned.loc[row, col], 2))
                dico[row].append((col, round(B_pruned.loc[row, col], 2)))

 
    plt.figure(figsize=(14, 8))
    pos = nx.spring_layout(G, k=2)
    nx.draw(G, pos, with_labels=True, node_color='lightblue', node_size=4000, font_size=9, arrowsize=25)
    edge_labels = nx.get_edge_attributes(G, 'weight')
    nx.draw_networkx_edge_labels(G, pos, edge_labels=edge_labels)
    plt.title("DAG")
    plt.tight_layout()
    plt.savefig('rex_causal_dag.png')


    return dico

lien= 'microdados_ed_basica_2024_SP.csv'
selected_cols = [
        'IN_AGUA_POTAVEL', 'IN_ENERGIA_REDE_PUBLICA', 'IN_INTERNET',
        'IN_LABORATORIO_INFORMATICA', 'QT_SALAS_UTILIZADAS', 'QT_MAT_BAS',  'QT_DOC_BAS']



test=graph(lien, selected_cols)
test

Tester si le modèle est cohérent en testant avec différentes colonnes


In [ ]:

selected_cols_altered = [
        'IN_AGUA_POTAVEL', 'IN_INTERNET',
        'IN_LABORATORIO_INFORMATICA', 'QT_SALAS_UTILIZADAS', 'QT_MAT_BAS',  'QT_DOC_BAS']



test_altered= graph(lien, selected_cols_altered)

test.pop('IN_ENERGIA_REDE_PUBLICA', None)





In [ ]:

test_altered

In [ ]:
test

ok. problème ici. en enlevant une variable inutile, on n'obtient pas le même graphe causal. mais on obtient les mêmes ancêtres (agua potavel et qt salas utilizadas)! on teste avec une variable qui a du sens, qu'on enlève dans les deux, pour voir si on trouve les mêmes ancêtres.

test sans in internet

In [ ]:

selected_cols_altered2 = [
        'IN_AGUA_POTAVEL',
        'IN_LABORATORIO_INFORMATICA', 'QT_SALAS_UTILIZADAS', 'QT_MAT_BAS',  'QT_DOC_BAS']



test_altered2= graph(lien, selected_cols_altered2)

test.pop('IN_INTERNET', None)

In [ ]:
test

In [ ]:
test_altered2

In [ ]:
selected_cols_altered3 = [
        'IN_AGUA_POTAVEL',
        'IN_LABORATORIO_INFORMATICA', 'QT_SALAS_UTILIZADAS', 'QT_MAT_BAS'  ]



test_altered2= graph(lien, selected_cols_altered3)

test.pop('QT_DOC_BAS', None)

en fait on retrouve toujours les mêmes parents absolus
idée: supprimer les parents absolus puis chercher la prochaine génération de parents absolus. maintenant on tente de supprimer un ancêtre qt salas

In [ ]:
selected_cols4 = [
        'IN_AGUA_POTAVEL', 'IN_ENERGIA_REDE_PUBLICA', 'IN_INTERNET',
        'IN_LABORATORIO_INFORMATICA', 'QT_MAT_BAS',  'QT_DOC_BAS']

test_altered4= graph(lien, selected_cols4)

selected_cols = [
        'IN_AGUA_POTAVEL', 'IN_ENERGIA_REDE_PUBLICA', 'IN_INTERNET',
        'IN_LABORATORIO_INFORMATICA', 'QT_SALAS_UTILIZADAS', 'QT_MAT_BAS',  'QT_DOC_BAS']



test=graph(lien, selected_cols)
test.pop('QT_SALAS_UTILIZADAS', None)


### on trouve pratiquement les mêmes résultats

on teste un bootstrap

In [ ]:

import pandas as pd
import numpy as np
from sklearn.linear_model import LinearRegression
import networkx as nx
from sklearn.impute import SimpleImputer
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler


def graph2(df, cols):


    imputer = SimpleImputer(strategy='median')
    X_clean = pd.DataFrame(imputer.fit_transform(df[cols]), columns=cols)
    scaler = StandardScaler()
    X_scaled = pd.DataFrame(scaler.fit_transform(X_clean), columns=cols)


    binary_data = X_clean[[c for c in cols if c.startswith('IN_')]]
    non_binary_data = X_clean[[c for c in cols if not c.startswith('IN_')]]


    #rex
    def rex_algorithm(data_df):
        X = data_df.values.copy()
        names = data_df.columns.tolist()
        n = len(names)
        remaining = list(range(n))
        order = []
        
        #iterate
        while len(remaining) > 0:
            scores = []
            for i in remaining:
                total_dep= 0
                xi = X[:, i].reshape(-1, 1)
                for j in remaining:
                    if i == j: continue
                    xj = X[:, j].reshape(-1, 1)
                    lr = LinearRegression().fit(xi,xj)
                    residual = xj - lr.predict(xi)
                    total_dep += abs(np.corrcoef(xi.flatten(), residual.flatten())[0,1])
                scores.append((total_dep, i))
            
            scores.sort()
            best_root = scores[0][1]
            order.append(best_root)
            remaining.remove(best_root)
            if len(remaining) >0:
                xi_root = X[:, best_root].reshape(-1, 1)
                for j in remaining:
                    xj = X[:,j].reshape(-1, 1)
                    lr = LinearRegression().fit(xi_root, xj)
                    X[:, j] = (xj - lr.predict(xi_root)).flatten()
                    
        return [names[i] for i in order]

    causal_order = rex_algorithm(X_scaled)

    B = pd.DataFrame(0.0, index=cols, columns=cols)
    X_std_df = pd.DataFrame(X_scaled, columns=cols)

    for i, target in enumerate(causal_order):
        if i == 0: continue
        parents = causal_order[:i]
        
        y = X_std_df[target]
        X_p = X_std_df[parents]
        
        reg = LinearRegression().fit(X_scaled[parents], X_scaled[target])
        for p_idx, p_name in enumerate(parents):
            B.loc[target, p_name] = reg.coef_[p_idx]


    B_pruned= B.copy()
    B_pruned[np.abs(B_pruned) < 0.05] =0

    dico = {node: [] for node in cols}
    G= nx.DiGraph()
    for node in cols:
        G.add_node(node)
        

    for row in B_pruned.index:
        for col in B_pruned.columns:
            if B_pruned.loc[row, col]!= 0:
                G.add_edge(col, row, weight=round(B_pruned.loc[row, col], 2))
                dico[row].append((col, round(B_pruned.loc[row, col], 2)))

    """""
    plt.figure(figsize=(14, 8))
    pos = nx.spring_layout(G, k=2)
    nx.draw(G, pos, with_labels=True, node_color='lightblue', node_size=4000, font_size=9, arrowsize=25)
    edge_labels = nx.get_edge_attributes(G, 'weight')
    nx.draw_networkx_edge_labels(G, pos, edge_labels=edge_labels)
    plt.title("DAG")
    plt.tight_layout()
    plt.savefig('rex_causal_dag.png')

    """""
    return dico



In [ ]:
from sklearn.utils import resample

def boostrap(lien, cols, n_iterations=50):
    df = pd.read_csv(lien, sep=';', encoding='latin1', low_memory=False)
    all_edges={}
    for i in range(n_iterations):
        df_sample = resample(df[cols], n_samples=len(df)//2) 
        current_dico = graph2(df_sample, cols) 
        for parent in current_dico.keys():
            for child, weight in current_dico[parent]:
                edge = (parent, child)
                if edge not in all_edges:
                    all_edges[edge] = []
                all_edges[edge].append(weight)

    robust_links = []
    for edge, weights in all_edges.items():
        confidence = len(weights) /n_iterations
        if confidence >= 0.40:
            avg_weight = np.mean(weights)
            robust_links.append({'cause': edge[0], 'effet': edge[1], 
                                 'confiance': confidence, 'poidsmoyen': avg_weight})
            
    return pd.DataFrame(robust_links)

res=boostrap(lien, selected_cols, n_iterations=50)
print(res)

In [ ]:
import networkx as nx
import matplotlib.pyplot as plt

def draw_robust_graph(df_robust):
    G = nx.DiGraph()
    for _, row in df_robust.iterrows():
        G.add_edge(
            row['cause'], 
            row['effet'], 
            weight=row['poidsmoyen'], 
            confiance=row['confiance']
        )

    fig, ax = plt.subplots(figsize=(15, 11))
    pos = nx.spring_layout(G, k=3, seed=42)
    nx.draw_networkx_nodes(G, pos, node_size=5000, node_color='white', edgecolors='black', linewidths=2, ax=ax)
    nx.draw_networkx_labels(G, pos, font_size=10, font_weight='bold', ax=ax)
    for u, v, d in G.edges(data=True):
        color = 'blue' if d['weight'] > 0 else 'red'
        width = abs(d['weight']) * 6 + 1
        alpha = d['confiance']

        rad = 0.2 if G.has_edge(v, u) else 0.0
        ax.annotate("",
                    xy=pos[v], xycoords='data',
                    xytext=pos[u], textcoords='data',
                    arrowprops=dict(
                        arrowstyle="->,head_width=0.5,head_length=0.8",
                        color=color,
                        shrinkA=30,
                        shrinkB=30,
                        connectionstyle=f"arc3,rad={rad}",
                        linewidth=width, 
                        alpha=alpha,
                        mutation_scale=30 ))

draw_robust_graph(res)


amélioration: vérifier que le graphe est fonctionnel

In [ ]:
from scipy.stats import pearsonr

def dico(res):
    res_dico={}
    for _, row in res.iterrows():
        cause = row['cause']
        effet = row['effet']
        confiance = round(float(row['confiance']), 2)
        if cause not in res_dico:
            res_dico[cause] = []
        res_dico[cause].append((effet, confiance))

def calculate_ace(df, treatment, outcome, controls, dico_robuste):
    y = df[outcome]
    X = df[[treatment] + controls]
    reg = LinearRegression().fit(X, y)
    return reg.coef_[0]

def test_d_separation(df, node_a, node_b, conditioning_set):
    if not conditioning_set:
        corr, p_val = pearsonr(df[node_a], df[node_b])
    else:
        y = df[node_a]
        X_a = df[conditioning_set]
        res_a = y - LinearRegression().fit(X_a, y).predict(X_a)
        
        z = df[node_b]
        X_b = df[conditioning_set]
        res_b = z - LinearRegression().fit(X_b, z).predict(X_b)
        
        corr, p_val = pearsonr(res_a, res_b)
    return corr, p_val


def analysis(df, cols, res):
    scaler = StandardScaler()
    df_scaled = pd.DataFrame(scaler.fit_transform(df[cols]), columns=cols)

    dico_robuste=dico(res)
    
    controls = [c for c in cols if c not in ['QT_SALAS_UTILIZADAS', 'QT_MAT_BAS']]
    ace_val = calculate_ace(df_scaled, 'QT_SALAS_UTILIZADAS', 'QT_MAT_BAS', controls, dico_robuste)
    print(f"ACE: {ace_val}")
    
    corr, p = test_d_separation(df_scaled, 'IN_AGUA_POTAVEL', 'QT_DOC_BAS', ['QT_MAT_BAS'])
    print(f"indep: corr={corr}, p={p}")
    

df = pd.read_csv('microdados_ed_basica_2024_SP.csv', sep=';', encoding='latin1', low_memory=False)
analysis(df, selected_cols, res)